# Run ALPR Vehicle System in Google Colab

Use this notebook to run the ALPR & Vehicle Classification dashboard on a Colab GPU runtime. 

**Recommended runtime:** Go to **Runtime → Change runtime type** and select **T4 GPU**.

In [ ]:
# 1) Verify GPU acceleration is active
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Type:', torch.cuda.get_device_name(0))

## 1) Clone repository from GitHub

Clone the project repository directly into the Colab environment.

In [ ]:
# 1) Clone repo
%cd /content
!rm -rf /content/ALPR-Vehicle-System

# Clone the repository
!git clone https://github.com/x0uls/ALPR-Vehicle-System.git

PROJECT_DIR = '/content/ALPR-Vehicle-System'
%cd $PROJECT_DIR
!ls -la

## Environment Setup & File Check

In [ ]:
# 2) Install Tesseract OCR & Python dependencies
!sudo apt-get update -qq
!sudo apt-get install -y tesseract-ocr tesseract-ocr-eng

# Install Python dependencies
!pip install -q ultralytics easyocr pandas openpyxl uvicorn fastapi pyngrok python-multipart jinja2 pytesseract supervision imutils deskew scikit-image jiwer

!tesseract --version
!tesseract --list-langs

In [ ]:
# 3) Verify required structure & cache models (pre-download)
from pathlib import Path
required_paths = [
    Path('app.py'),
    Path('src/pipeline.py'),
    Path('src/templates/index.html'),
    Path('models/yolo_plate/best.pt')
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required project files:\n' + '\n'.join(missing))

print('Downloading base YOLOv8 model if not cached...')
if not Path('yolov8n.pt').exists():
    from ultralytics import YOLO
    YOLO('yolov8n.pt')

print('Downloading EasyOCR models (detection/recognition) if not cached...')
import easyocr
import torch
easyocr.Reader(['en'], gpu=torch.cuda.is_available())

print('✅ Project structure ready and all models cached.')

## Launch Interactive Web Dashboard

In [ ]:
#@title 4) Start Server & Open Public Ngrok Tunnel
NGROK_AUTH_TOKEN = "" #@param {type:"string"}

import threading, time, os, subprocess

PORT = 7860

# Clear previous logs
!rm -f uvicorn.log

# Run uvicorn in background thread and redirect output to uvicorn.log
def _run():
    os.system(f'python -m uvicorn app:app --host 0.0.0.0 --port {PORT} > uvicorn.log 2>&1')

threading.Thread(target=_run, daemon=True).start()
time.sleep(5) # Give the server a few seconds to initialize

if not NGROK_AUTH_TOKEN.strip():
    print("❌ Error: Ngrok requires a free auth token.\n👉 Get one at: https://dashboard.ngrok.com/get-started/your-authtoken\nThen copy-paste it into the NGROK_AUTH_TOKEN field and re-run this cell.")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
    try:
        ngrok.kill()
    except:
        pass
    tunnel = ngrok.connect(PORT)
    print(f'\n✅ Web Dashboard (Ngrok): {tunnel.public_url}')
    print('\n💡 Note: Ngrok tunnels do not have a timeout limit for long processing runs.')
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        ngrok.disconnect(tunnel.public_url)
        print('Ngrok tunnel shut down.')